In [1]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
    print('GPU:', torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print('CPU')

GPU: Tesla P100-PCIE-16GB


In [2]:
#import the used libraries
import numpy as np


In [3]:
size = 2
num_of_matrecies = 46000


np.random.seed(42)
A_matrecies = np.random.uniform(-1, 1, size=(num_of_matrecies, size, size))

np.random.seed(43)
B_matrecies = np.random.uniform(-1, 1, size=(num_of_matrecies, size, size))

C_matrecies = np.zeros((num_of_matrecies, size, size))
for i in range (num_of_matrecies):
    C_matrecies[i] = A_matrecies[i] @ B_matrecies[i]

In [4]:
# flattening
A_matrecies = A_matrecies.reshape(46000,-1)
B_matrecies = B_matrecies.reshape(46000,-1)
C_matrecies = C_matrecies.reshape(46000,-1)

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

In [6]:
# Splitting (70% for train and 30 for test) and converting the dataframe to a tensor
A_train  = torch.tensor(A_matrecies[:32000])
B_train = torch.tensor(B_matrecies[:32000])
C_train = torch.tensor(C_matrecies[:32000])

A_test = torch.tensor(A_matrecies[32000:])
B_test = torch.tensor(B_matrecies[32000:])
C_test = torch.tensor(C_matrecies[32000:])

In [7]:
class MultiplyActivation(nn.Module):
    def __init__(self):
        super(MultiplyActivation, self).__init__()

    def forward(self, input_set1, input_set2):
        # Element-wise multiplication of the two sets of neurons
        result = input_set1 * input_set2

        return result

In [8]:
mat_size= size*size
mul = 7

# Define the neural network architecture
class SimpleANN(nn.Module):
    def __init__(self):
        super(SimpleANN, self).__init__()
        self.input_a = nn.Linear(mat_size, mul,bias=False)  # Input layer to hidden layer
        self.input_b = nn.Linear(mat_size, mul,bias=False)  # Input layer to hidden layer

        self.output_c = nn.Linear(mul, mat_size,bias=False)  # Hidden layer to output layer

        
        self.multiply_activation = MultiplyActivation()
        
        nn.init.xavier_uniform_(self.input_a.weight)
        nn.init.xavier_uniform_(self.input_b.weight)
        nn.init.xavier_uniform_(self.output_c.weight)


    def forward(self, a,b):
        a,b = self.input_a(a), self.input_b(b)
        
        x = self.multiply_activation(a,b)
        
        x = self.output_c(x)

        return x
    

model = SimpleANN()

criterion = nn.MSELoss() 

optimizer = optim.Adam(model.parameters(), lr=0.05)


In [9]:
if torch.cuda.is_available():
    model = model.cuda()
    criterion = criterion.cuda()

In [10]:
from torch.utils.data import DataLoader, TensorDataset
train_dataset = TensorDataset(A_train, B_train, C_train)
train_loader = DataLoader(train_dataset, batch_size=32000, shuffle=True)


test_dataset = TensorDataset(A_test, B_test , C_test)
test_loader = DataLoader(test_dataset, batch_size=14000, shuffle=True)

In [11]:
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
import os

/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.3
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [18]:
lo = []
epo = []

In [17]:
model.load_state_dict(torch.load('/kaggle/input/2by2initialisation-for-comparaison/2_2_7perceptrons_initialisation.pth'))


<All keys matched successfully>

In [16]:
def regularisation(model, alpha):
    somme =0

    for param_tensor in model.state_dict():

        #L0 regularisation for each weight
        l0 = abs(model.state_dict()[param_tensor].data)
        

        #L1 regularisation for each weight
        l1 = abs(model.state_dict()[param_tensor].data - 1)

        #L -1 regularisation for each weight
        l_neg1 = abs(model.state_dict()[param_tensor].data + 1)
        
        min_values,_ = torch.min(torch.stack((alpha *l0,l1,l_neg1),dim = 2), dim=2)
        somme += torch.sum(min_values)


    total = sum(p.numel() for p in model.parameters())
    return somme * (torch.tensor(1/total))

In [1]:
def zeroVectors(model, mul):

    ac = np.ones(mul) 

    for i in range (mul):
        if( ((torch.all(torch.abs(model.state_dict()['input_a.weight'][i]))) <= 0.5) or ((torch.all(abs(model.state_dict()['input_b.weight'][i]))) <= 0.5)):
            ac[i] = 0

    return (torch.tensor((sum(ac))/mul))

In [ ]:
# Flatten parameters
flat_parameters = torch.cat([param.view(-1) for param in model.parameters()])
print(flat_parameters)

In [ ]:
# Now, reflatten the modified parameters
start_idx = 0
for param in model.parameters():
    end_idx = start_idx + param.numel()  # Get the index of the end of the parameter
    param.data = flat_parameters[start_idx:end_idx].view(param.shape)  # Reshape the flattened parameters
    start_idx = end_idx  # Update the start index for the next parameter
    
    

In [15]:
for name, param in model.named_parameters():
    print("Parameter:", name)
    print("Values:", param.data)

Parameter: input_a.weight
Values: tensor([[-0.3476,  0.8942,  0.4051, -0.6265],
        [ 0.9196, -0.2356,  0.9060, -0.8546],
        [ 0.8374, -0.4757, -0.0519,  0.2015],
        [ 0.4987,  0.4629, -0.7320,  0.5248],
        [ 0.1621,  0.1560,  0.1334,  0.3110],
        [-0.5965,  0.2171,  0.2175,  0.7465],
        [-0.2466, -0.9925, -0.5077,  0.2899]], device='cuda:0')
Parameter: input_b.weight
Values: tensor([[-0.2127, -0.4873,  0.2448,  0.9164],
        [-0.5867,  0.2136, -0.9807, -0.7450],
        [-0.7978,  0.0470, -0.8197,  0.5932],
        [-0.0182,  0.2935,  0.0481,  0.5718],
        [ 0.9660,  0.1435, -0.3599, -0.8950],
        [-0.0781, -0.9706,  0.3212,  0.0902],
        [ 0.0442, -0.3843, -0.8278, -0.3930]], device='cuda:0')
Parameter: output_c.weight
Values: tensor([[ 0.7939,  0.6565, -0.0586,  0.2403,  0.8694,  0.6254, -0.9625],
        [-0.9699,  0.0915,  0.4341, -0.5673,  0.3753, -0.9674, -0.3275],
        [-0.2177, -0.0342,  0.5799, -0.7409, -0.4981,  0.6765,  0.4252]

In [ ]:
epochs = 10000   # Adjust the number of epochs as needed
alpha = 1/2


for epoch in range(epochs):
    total_loss = 0.0

    for inputs,inputs_, targets in train_loader:
        if torch.cuda.is_available():
            inputs = inputs.cuda()
            inputs_ = inputs_.cuda()
            targets = targets.cuda()

        # Zero the gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs.float(),inputs_.float())  # Assuming your inputs are floats

        # Compute the loss
        loss = criterion(outputs, targets.float()) + 2 * regularisation(model, alpha)
        print(f'MSE= {criterion(outputs, targets.float())} and regularisation= {regularisation(model,alpha)}')
        total_loss += loss.item()

        # Backward pass
        loss.backward()

        # Update the weights
        optimizer.step()


    # Print the training loss for each epoch
    print(f"Epoch {epoch + 1}/{epochs}, Loss: {total_loss/len(train_loader)}\n")
    epo.append(epoch)
    lo.append(total_loss/len(train_loader))
    
    
    if ((epoch+1) % 500 == 0):
        save_dir = '/kaggle/working/'
        torch.save(model.state_dict(),os.path.join(save_dir, f'2*2_8perceptrons_2MinReg_Alpha0.5_{epoch+1}epochs.pth'))
        




MSE= 0.2481124848127365 and regularisation= 0.17530620098114014
Epoch 1/10000, Loss: 0.598724901676178

MSE= 0.23300933837890625 and regularisation= 0.17265509068965912
Epoch 2/10000, Loss: 0.5783195495605469

MSE= 0.2139957696199417 and regularisation= 0.16874366998672485
Epoch 3/10000, Loss: 0.5514830946922302

MSE= 0.19476664066314697 and regularisation= 0.1649288535118103
Epoch 4/10000, Loss: 0.5246243476867676

MSE= 0.17689423263072968 and regularisation= 0.16095119714736938
Epoch 5/10000, Loss: 0.49879664182662964

MSE= 0.16064020991325378 and regularisation= 0.1574029177427292
Epoch 6/10000, Loss: 0.47544604539871216

MSE= 0.14571142196655273 and regularisation= 0.15405623614788055
Epoch 7/10000, Loss: 0.45382389426231384

MSE= 0.1317208707332611 and regularisation= 0.15275098383426666
Epoch 8/10000, Loss: 0.43722283840179443

MSE= 0.11839717626571655 and regularisation= 0.151785746216774
Epoch 9/10000, Loss: 0.4219686686992645

MSE= 0.10557504743337631 and regularisation= 0.150

In [ ]:
import csv
# Combine the lists into a list of tuples
data = list(zip(epo, lo))

# Specify the file path
csv_file_path = '/kaggle/working/loss_epochs_.csv'

with open(csv_file_path, 'w', newline='') as csv_file:
    writer = csv.writer(csv_file)
    
    writer.writerow(['epochs', 'loss'])
    
    writer.writerows(data)

In [63]:
model.load_state_dict(torch.load('/kaggle/working/2*2_8perceptrons_2MinReg_Alpha0.5_10000epochs.pth'))

<All keys matched successfully>

In [ ]:
# Evaluate the model on the test set
model.eval() 
test_loss = 0.0

with torch.no_grad():
    for inputs,inputs_, targets in test_loader:
        if torch.cuda.is_available():
            inputs = inputs.cuda()
            inputs_ = inputs_.cuda()
            targets = targets.cuda()
        
        
        outputs = model(inputs.float(),inputs_.float())  
        test_loss += criterion(outputs, targets.float()).item()

# Calculate the average test loss
average_test_loss = test_loss / len(test_loader)
print(f"Average Test Loss: {average_test_loss}")

In [64]:
#rounded_model = SimpleANN()
# Copy the parameters from the original model to the copied model
#rounded_model.load_state_dict(model.state_dict())

for param_tensor in model.state_dict():
    model.state_dict()[param_tensor].data.round_()

In [ ]:
for name, param in model.state_dict().items():
    print(f"Parameter name: {name}, Size: {param.size()}")
    print(f"Parameter values:\n{param.data}\n")


In [ ]:
model.state_dict()['input_a.weight'][0,0] = 1
model.state_dict()['input_a.weight'][0,1] = 0
model.state_dict()['input_a.weight'][0,2] = 0
model.state_dict()['input_a.weight'][0,3] = 1

model.state_dict()['input_a.weight'][1,0] = 0
model.state_dict()['input_a.weight'][1,1] = 0
model.state_dict()['input_a.weight'][1,2] = 1
model.state_dict()['input_a.weight'][1,3] = 1

model.state_dict()['input_a.weight'][2,0] = 1
model.state_dict()['input_a.weight'][2,1] = 0
model.state_dict()['input_a.weight'][2,2] = 0
model.state_dict()['input_a.weight'][2,3] = 0

model.state_dict()['input_a.weight'][3,0] = 0
model.state_dict()['input_a.weight'][3,1] = 0
model.state_dict()['input_a.weight'][3,2] = 0
model.state_dict()['input_a.weight'][3,3] = 1

model.state_dict()['input_a.weight'][4,0] = 1
model.state_dict()['input_a.weight'][4,1] = 1
model.state_dict()['input_a.weight'][4,2] = 0
model.state_dict()['input_a.weight'][4,3] = 0

model.state_dict()['input_a.weight'][5,0] = -1
model.state_dict()['input_a.weight'][5,1] = 0
model.state_dict()['input_a.weight'][5,2] = 1
model.state_dict()['input_a.weight'][5,3] = 0

model.state_dict()['input_a.weight'][6,0] = 0
model.state_dict()['input_a.weight'][6,1] = 1
model.state_dict()['input_a.weight'][6,2] = 0
model.state_dict()['input_a.weight'][6,3] = -1



In [ ]:
model.state_dict()['input_b.weight'][0,0] = 1
model.state_dict()['input_b.weight'][0,1] = 0
model.state_dict()['input_b.weight'][0,2] = 0
model.state_dict()['input_b.weight'][0,3] = 1

model.state_dict()['input_b.weight'][1,0] = 1
model.state_dict()['input_b.weight'][1,1] = 0
model.state_dict()['input_b.weight'][1,2] = 0
model.state_dict()['input_b.weight'][1,3] = 0

model.state_dict()['input_b.weight'][2,0] = 0
model.state_dict()['input_b.weight'][2,1] = 1
model.state_dict()['input_b.weight'][2,2] = 0
model.state_dict()['input_b.weight'][2,3] = -1

model.state_dict()['input_b.weight'][3,0] = -1
model.state_dict()['input_b.weight'][3,1] = 0
model.state_dict()['input_b.weight'][3,2] = 1
model.state_dict()['input_b.weight'][3,3] = 0

model.state_dict()['input_b.weight'][4,0] = 0
model.state_dict()['input_b.weight'][4,1] = 0
model.state_dict()['input_b.weight'][4,2] = 0
model.state_dict()['input_b.weight'][4,3] = 1

model.state_dict()['input_b.weight'][5,0] = 1
model.state_dict()['input_b.weight'][5,1] = 1
model.state_dict()['input_b.weight'][5,2] = 0
model.state_dict()['input_b.weight'][5,3] = 0

model.state_dict()['input_b.weight'][6,0] = 0
model.state_dict()['input_b.weight'][6,1] = 0
model.state_dict()['input_b.weight'][6,2] = 1
model.state_dict()['input_b.weight'][6,3] = 1



In [ ]:
model.state_dict()['output_c.weight'][0,0] = 1
model.state_dict()['output_c.weight'][0,1] = 0
model.state_dict()['output_c.weight'][0,2] = 0
model.state_dict()['output_c.weight'][0,3] = 1
model.state_dict()['output_c.weight'][0,4] = -1
model.state_dict()['output_c.weight'][0,5] = 0
model.state_dict()['output_c.weight'][0,6] = 1

model.state_dict()['output_c.weight'][1,0] = 0
model.state_dict()['output_c.weight'][1,1] = 0
model.state_dict()['output_c.weight'][1,2] = 1
model.state_dict()['output_c.weight'][1,3] = 0
model.state_dict()['output_c.weight'][1,4] = 1
model.state_dict()['output_c.weight'][1,5] = 0
model.state_dict()['output_c.weight'][1,6] = 0

model.state_dict()['output_c.weight'][2,0] = 0
model.state_dict()['output_c.weight'][2,1] = 1
model.state_dict()['output_c.weight'][2,2] = 0
model.state_dict()['output_c.weight'][2,3] = 1
model.state_dict()['output_c.weight'][2,4] = 0
model.state_dict()['output_c.weight'][2,5] = 0
model.state_dict()['output_c.weight'][2,6] = 0

model.state_dict()['output_c.weight'][3,0] = 1
model.state_dict()['output_c.weight'][3,1] = -1
model.state_dict()['output_c.weight'][3,2] = 1
model.state_dict()['output_c.weight'][3,3] = 0
model.state_dict()['output_c.weight'][3,4] = 0
model.state_dict()['output_c.weight'][3,5] = 1
model.state_dict()['output_c.weight'][3,6] = 0

In [ ]:
print(model.state_dict())